In [ ]:
import sys
import os

# Add current directory to path so we can import glow
sys.path.append(os.getcwd())

from glow.data_loading.loader import DataLoader
from glow.gating.manager import GatingManager

In [2]:
# 1. Setup Data Loader
loader = DataLoader()

# Define your input directory here. 
# Replace with your actual path, e.g. "/home/xwz/Projects/Flow/flow-data/dhy_20251001_fcs/"
input_dir = "/home/xwz/Projects/Flow/flow-data/dhy_20251001_fcs/"

print(f"Scanning {input_dir}...")
files = loader.scan_directory(input_dir)
print(f"Found {len(files)} files.")
if files:
    print("Sample:", files[:3])

Scanning /home/xwz/Projects/Flow/flow-data/dhy_20251001_fcs/...
Found 10 files.
Sample: ['/home/xwz/Projects/Flow/flow-data/dhy_20251001_fcs/D2.fcs', '/home/xwz/Projects/Flow/flow-data/dhy_20251001_fcs/C2.fcs', '/home/xwz/Projects/Flow/flow-data/dhy_20251001_fcs/A1.fcs']


In [3]:
# 2. Metadata & Renaming Preview
# This step generates the proposed names (foldername_filename)
preview = loader.generate_preview_metadata(files, rename_strategy='folder_name')

import pandas as pd
pd.DataFrame(preview)[['display_name', 'original_path', 'condition']].head()

,display_name,original_path,condition
0,dhy_20251001_fcs_D2.fcs,/home/xwz/Projects/Flow/flow-data/dhy_20251001...,dhy_20251001_fcs
1,dhy_20251001_fcs_C2.fcs,/home/xwz/Projects/Flow/flow-data/dhy_20251001...,dhy_20251001_fcs
2,dhy_20251001_fcs_A1.fcs,/home/xwz/Projects/Flow/flow-data/dhy_20251001...,dhy_20251001_fcs
3,dhy_20251001_fcs_E1.fcs,/home/xwz/Projects/Flow/flow-data/dhy_20251001...,dhy_20251001_fcs
4,dhy_20251001_fcs_B1.fcs,/home/xwz/Projects/Flow/flow-data/dhy_20251001...,dhy_20251001_fcs


In [4]:
# 3. Process Files (Copy/Rename)
# This copies files to a local 'debug_processed' folder to avoid touching originals
processed_meta = loader.process_files(preview, output_dir='debug_processed')
print("Files processed.")

Files processed.


In [5]:
# 3. Simulate UI Metadata Enrichment
# Let's manually inject the metadata that causes the crash (e.g. inconsistent types or new columns)
enriched_meta = []
for i, item in enumerate(preview):
    new_item = item.copy()
    # Simulate user adding columns in UI
    new_item['Sample'] = new_item['display_name'] # Unique ID
    new_item['Replicate'] = "1" 
    new_item['Cell_Line'] = "HEK293" if i % 2 == 0 else "Jurkat"
    new_item['Experiment_Name'] = "Test_Exp"
    
    # Simulate a potentially problematic scenario: missing values or different types?
    # Cytoflow usually hates it if one tube has a condition and another doesn't, 
    # OR if types differ.
    if i == 0:
        new_item['Timepoint'] = 24 # Int
    else:
        new_item['Timepoint'] = "48" # String - this mixed type often kills it.
        
    enriched_meta.append(new_item)

print("Enriched Metadata Sample:", enriched_meta[0])

# Process Files (Standard step)
processed_meta = loader.process_files(enriched_meta, output_dir='debug_processed')
print("Files processed.")

Enriched Metadata Sample: {'original_path': '/home/xwz/Projects/Flow/flow-data/dhy_20251001_fcs/D2.fcs', 'display_name': 'dhy_20251001_fcs_D2.fcs', 'condition': 'dhy_20251001_fcs', 'Sample': 'dhy_20251001_fcs_D2.fcs', 'Replicate': '1', 'Cell_Line': 'HEK293', 'Experiment_Name': 'Test_Exp', 'Timepoint': 24}
Files processed.


In [8]:
# 4. Debug Loading Logic
import importlib
import base_pipeline.data_loader
importlib.reload(base_pipeline.data_loader)
from base_pipeline.data_loader import DataLoader
loader = DataLoader() # Re-init

try:
    print("Attempting to load experiment with mixed metadata...")
    # Using the same enriched_meta from previous cell
    processed_meta = loader.process_files(enriched_meta, output_dir='debug_processed_v2')
    experiment = loader.load_as_cytoflow_experiment(processed_meta)
    print("Success! Experiment loaded.")
    
    # experiment.conditions is a dictionary of {condition_name: type}, not a DataFrame.
    print("\nCondition Keys:", experiment.conditions.keys())
    
    # Check actual channels 
    print("\nAvailable Channels:", experiment.channels)
    
    # To see the actual data with attached conditions:
    print("\nFirst 5 rows of Experiment Data (including conditions):")
    print(experiment.data.head())
    
except Exception as e:
    print(f"FAILED: {e}")
    import traceback
    traceback.print_exc()

Attempting to load experiment with mixed metadata...
Success! Experiment loaded.

Condition Keys: dict_keys(['Cell_Line', 'Condition', 'Experiment_Name', 'Replicate', 'Sample', 'Timepoint'])

Available Channels: ['Event Info', 'FSC 488/10-A', 'FSC 488/10-H', 'FSC 488/10-W', 'GFP-A', 'GFP-H', 'GFP-W', 'SSC 488/10-A', 'SSC 488/10-H', 'SSC 488/10-W', 'TIME', 'TLSW', 'TMSW', 'TagBFP-A', 'TagBFP-H', 'TagBFP-W', 'mCherry-A', 'mCherry-H', 'mCherry-W']

First 5 rows of Experiment Data (including conditions):
  Cell_Line         Condition  Event Info Experiment_Name  FSC 488/10-A  \
0    HEK293  dhy_20251001_fcs  2147418112        Test_Exp      18636032   
1    HEK293  dhy_20251001_fcs  2147418112        Test_Exp      38732288   
2    HEK293  dhy_20251001_fcs  2147418112        Test_Exp      19091968   
3    HEK293  dhy_20251001_fcs  2147418112        Test_Exp      45036032   
4    HEK293  dhy_20251001_fcs  2147418112        Test_Exp      20742144   

   FSC 488/10-H  FSC 488/10-W  GFP-A  GFP-H

/home/dhy/miniconda3/envs/glow_env/lib/python3.12/site-packages/cytoflow/operations/import_op.py:420: CytoflowOpWarning: The data range $PnR doesn't match the data bits $PnB for channel TLSW, masking out 1 bits
/home/dhy/miniconda3/envs/glow_env/lib/python3.12/site-packages/cytoflow/operations/import_op.py:420: CytoflowOpWarning: The data range $PnR doesn't match the data bits $PnB for channel TMSW, masking out 1 bits
/home/dhy/miniconda3/envs/glow_env/lib/python3.12/site-packages/cytoflow/operations/import_op.py:420: CytoflowOpWarning: The data range $PnR doesn't match the data bits $PnB for channel Event Info, masking out 1 bits
/home/dhy/miniconda3/envs/glow_env/lib/python3.12/site-packages/cytoflow/operations/import_op.py:420: CytoflowOpWarning: The data range $PnR doesn't match the data bits $PnB for channel FSC 488/10-H, masking out 1 bits
/home/dhy/miniconda3/envs/glow_env/lib/python3.12/site-packages/cytoflow/operations/import_op.py:420: CytoflowOpWarning: The data range $PnR d

In [ ]:
# 5. Live Gating with Correct Channels
# Inspect channels from the previous cell output to pick correct ones.
# Typically: 'FSC 488/10-A', 'SSC 488/10-A' for typical Cytometers
if 'experiment' in locals() and experiment:
    from base_pipeline.gating import GatingManager
    gater = GatingManager()
    
    # Let's try to guess good channels or default to first two
    x_chan = "FSC 488/10-A" if "FSC 488/10-A" in experiment.channels else experiment.channels[0]
    y_chan = "SSC 488/10-A" if "SSC 488/10-A" in experiment.channels else experiment.channels[1]
    
    print(f"Gating on {x_chan} vs {y_chan}...")
    
    try:
        experiment = gater.auto_gate_live_cells(experiment, x_channel=x_chan, y_channel=y_chan)
        print("Auto gating complete.")
        print("New Conditions:", experiment.conditions.keys())
        print(experiment.data[[x_chan, y_chan, 'AutoLiveGate']].head())
    except Exception as e:
        print(f"Gating error: {e}")
        import traceback
        traceback.print_exc()

In [9]:
import active_learning.storage as storage
import os

# Check if file exists
print(os.listdir("active_learning_data"))


['feedback_20260107.jsonl']


In [ ]:
from active_learning.learner import ActiveLearner
from active_learning.storage import ActiveLearningLogger
import shutil
import os

# Setup test
test_storage = "test_al_data"
if os.path.exists(test_storage):
    shutil.rmtree(test_storage)
    
logger = ActiveLearningLogger(storage_dir=test_storage)

# Simulate User Feedback
# Cell Line A: Live Gate (5 points)
logger.log_gate_feedback(
    sample_name="Sample1",
    cell_line="CellLineA",
    x_channel="FSC-A",
    y_channel="SSC-A",
    gate_type="Polygon",
    gate_params=[(1,1), (2,1), (2,2), (1.5, 3), (1,2)], # 5 points
    user_decision="accepted"
)

# Cell Line A: Singlet Gate (4 points)
logger.log_gate_feedback(
    sample_name="Sample1",
    cell_line="CellLineA",
    x_channel="FSC-A",
    y_channel="FSC-H",
    gate_type="Polygon",
    gate_params=[(1,1), (10,1), (10,10), (1,10)], # 4 points
    user_decision="accepted"
)

# Initialize Learner
learner = ActiveLearner(storage_dir=test_storage, model_filename="test_weights.json")
learner.train()

# Check Predictions
live_pred = learner.predict("CellLineA", "Live")
singlet_pred = learner.predict("CellLineA", "Singlet")

print("Live Prediction:", live_pred)
print("Singlet Prediction:", singlet_pred)

# Verify persistence
learner.save_weights()
assert os.path.exists(os.path.join(test_storage, "test_weights.json"))

learner2 = ActiveLearner(storage_dir=test_storage, model_filename="test_weights.json")
print("Reloaded Prediction:", learner2.predict("CellLineA", "Live"))


In [ ]:
# Just checking imports for UI
try:
    from active_learning.learner import ActiveLearner
    print("Import successful")
except ImportError as e:
    print(e)
